# Cortex Search: BYO Embeddings & Cost Optimization

This lab validates the cost behavior of a Cortex Search Service built with user-provided (BYO) embeddings. You will create a multi-index service, verify that incremental refresh works correctly, measure cost, add data incrementally, and test optimization levers.

### What You'll Learn

- How to create a Cortex Search Service with BYO embeddings (zero EMBED_TEXT cost)
- How incremental refresh processes only new/changed rows
- How to read cost breakdowns from `CORTEX_SEARCH_DAILY_USAGE_HISTORY`
- What triggers a full rebuild (anti-patterns to avoid)
- How to apply optimization levers: `AUTO_SUSPEND`, `TARGET_LAG`, corpus pruning

**Prerequisite:** Run `lab/setup.sql` and then `lab/data_gen.py` before starting this notebook.

---
## Section 1: Connect & Explore the Data

In [ ]:
from snowflake.snowpark.context import get_active_session

session = get_active_session()
session.sql("USE DATABASE CORTEX_SEARCH_COST_DEMO").collect()
session.sql("USE SCHEMA DEMO").collect()
session.sql("USE WAREHOUSE CORTEX_SEARCH_COST_WH").collect()
print("Connected to CORTEX_SEARCH_COST_DEMO.DEMO")

In [ ]:
# Verify DOCUMENT_CHUNKS was loaded with embeddings
session.sql("""
    SELECT
        COUNT(*) AS TOTAL_CHUNKS,
        COUNT(EMBEDDING) AS WITH_EMBEDDING,
        COUNT(DISTINCT DOC_ID) AS DISTINCT_DOCS
    FROM DOCUMENT_CHUNKS
""").show()

In [ ]:
# Preview a few rows
session.sql("""
    SELECT CHUNK_ID, DOC_ID, DOC_TYPE, CATEGORY, LEFT(CHUNK_TEXT, 120) AS TEXT_PREVIEW,
           ARRAY_SIZE(EMBEDDING::ARRAY) AS EMBED_DIMS
    FROM DOCUMENT_CHUNKS
    LIMIT 5
""").show()

---
## Section 2: Create the Cortex Search Service with BYO Embeddings

We use the multi-index syntax:
- `TEXT INDEXES (CHUNK_TEXT)` -- enables keyword (lexical) search
- `VECTOR INDEXES (EMBEDDING(query_model='snowflake-arctic-embed-m-v1.5'))` -- BYO vectors at index time (zero embedding cost), Snowflake embeds query text at search time
- `PRIMARY KEY (CHUNK_ID)` -- enables optimized incremental refresh
- `REFRESH_MODE = INCREMENTAL` (default)

Snowflake will **not** call EMBED_TEXT on user-provided vectors at index time. The `query_model` only applies at search time to embed the query text.

In [ ]:
# Create the Cortex Search Service
session.sql("""
    CREATE OR REPLACE CORTEX SEARCH SERVICE DOCUMENT_SEARCH_SVC
        TEXT INDEXES (CHUNK_TEXT)
        VECTOR INDEXES (EMBEDDING(query_model='snowflake-arctic-embed-m-v1.5'))
        PRIMARY KEY (CHUNK_ID)
        ATTRIBUTES DOC_TYPE, CATEGORY, FILE_NAME
        WAREHOUSE = CORTEX_SEARCH_COST_WH
        TARGET_LAG = '1 hour'
        REFRESH_MODE = INCREMENTAL
        REQUEST_LOGGING = TRUE
    AS (
        SELECT
            CHUNK_ID,
            CHUNK_TEXT,
            EMBEDDING,
            DOC_TYPE,
            CATEGORY,
            FILE_NAME,
            DOC_ID,
            PAGE_NUMBER
        FROM DOCUMENT_CHUNKS
    )
""").collect()
print("Service created. Initial index build may take a minute...")

In [ ]:
# Inspect the service
session.sql("DESCRIBE CORTEX SEARCH SERVICE DOCUMENT_SEARCH_SVC").show()

In [ ]:
# Quick search test (query text is embedded by query_model at search time)
session.sql("""
    SELECT
        value['CHUNK_ID']::TEXT AS CHUNK_ID,
        value['DOC_TYPE']::TEXT AS DOC_TYPE,
        value['CATEGORY']::TEXT AS CATEGORY,
        LEFT(value['CHUNK_TEXT']::TEXT, 110) AS CHUNK_TEXT_PREVIEW
    FROM TABLE(FLATTEN(PARSE_JSON(
        SNOWFLAKE.CORTEX.SEARCH_PREVIEW(
            'CORTEX_SEARCH_COST_DEMO.DEMO.DOCUMENT_SEARCH_SVC',
            '{
                "query": "encryption key rotation compliance",
                "columns": ["CHUNK_ID", "DOC_TYPE", "CATEGORY", "CHUNK_TEXT"],
                "filter": {"@eq": {"CATEGORY": "Security"}},
                "limit": 3
            }'
        )
    )['results']))
""").show()

---
## Section 3: Baseline Cost Measurement

Query `CORTEX_SEARCH_DAILY_USAGE_HISTORY` to see the cost breakdown. For BYO embeddings, `EMBED_TEXT_TOKENS` credits should be **zero**.

> **Note:** This view has up to 24h latency. If the service was just created, you may see no rows yet. The DESCRIBE output above shows the service is active.

In [ ]:
# Daily cost breakdown by consumption type
session.sql("""
    SELECT
        USAGE_DATE,
        SERVICE_NAME,
        CONSUMPTION_TYPE,
        CREDITS,
        MODEL_NAME,
        TOKENS
    FROM SNOWFLAKE.ACCOUNT_USAGE.CORTEX_SEARCH_DAILY_USAGE_HISTORY
    WHERE SERVICE_NAME = 'DOCUMENT_SEARCH_SVC'
    ORDER BY USAGE_DATE DESC
    LIMIT 20
""").show()

In [ ]:
# Estimate serving cost from indexed data size
session.sql("""
    SELECT
        COUNT(*) AS TOTAL_ROWS,
        ROUND(COUNT(*) * (768 * 4 + 1000) / 1e9, 4) AS ESTIMATED_INDEXED_GB,
        ROUND(6.3 * COUNT(*) * (768 * 4 + 1000) / 1e9, 2) AS ESTIMATED_MONTHLY_SERVING_CR
    FROM DOCUMENT_CHUNKS
""").show()

---
## Section 4: Validate Incremental Refresh (Add New Documents)

We will INSERT 500 new chunks (50 new documents x 10 chunks each) with pre-computed embeddings, then verify that only the delta was processed -- not a full rebuild.

In [ ]:
# Record baseline row count
baseline = session.sql("SELECT COUNT(*) AS CNT FROM DOCUMENT_CHUNKS").collect()[0]["CNT"]
print(f"Baseline row count: {baseline}")

In [ ]:
# Insert 500 new chunks with embeddings (50 new docs x 10 chunks)
session.sql("""
    INSERT INTO DOCUMENT_CHUNKS (CHUNK_ID, DOC_ID, CHUNK_TEXT, DOC_TYPE, CATEGORY, FILE_NAME, PAGE_NUMBER, EMBEDDING)
    WITH docs AS (
        SELECT SEQ4() AS doc_num FROM TABLE(GENERATOR(ROWCOUNT => 50))
    ),
    chunks AS (
        SELECT SEQ4() AS chunk_num FROM TABLE(GENERATOR(ROWCOUNT => 10))
    )
    SELECT
        'NEW_DOC_' || LPAD(d.doc_num::VARCHAR, 4, '0') || '_CHUNK_' || LPAD(c.chunk_num::VARCHAR, 2, '0') AS CHUNK_ID,
        'NEW_DOC_' || LPAD(d.doc_num::VARCHAR, 4, '0') AS DOC_ID,
        'This is a newly added document chunk about ' ||
            ARRAY_CONSTRUCT('security policies', 'data governance', 'API integration',
                'performance tuning', 'incident response')[UNIFORM(0, 4, RANDOM())]::VARCHAR ||
            ' for the ' ||
            ARRAY_CONSTRUCT('Engineering', 'Legal', 'HR', 'Product', 'Security')[UNIFORM(0, 4, RANDOM())]::VARCHAR ||
            ' team. This content was added incrementally to test refresh behavior.' AS CHUNK_TEXT,
        ARRAY_CONSTRUCT('policy_guide', 'faq', 'procedure')[UNIFORM(0, 2, RANDOM())]::VARCHAR AS DOC_TYPE,
        ARRAY_CONSTRUCT('Engineering', 'Legal', 'HR', 'Product', 'Security')[UNIFORM(0, 4, RANDOM())]::VARCHAR AS CATEGORY,
        'docs/new/NEW_DOC_' || LPAD(d.doc_num::VARCHAR, 4, '0') || '.pdf' AS FILE_NAME,
        c.chunk_num + 1 AS PAGE_NUMBER,
        SNOWFLAKE.CORTEX.EMBED_TEXT_768(
            'snowflake-arctic-embed-m-v1.5',
            'This is a newly added document chunk about ' ||
                ARRAY_CONSTRUCT('security policies', 'data governance', 'API integration',
                    'performance tuning', 'incident response')[UNIFORM(0, 4, RANDOM())]::VARCHAR ||
                ' for the ' ||
                ARRAY_CONSTRUCT('Engineering', 'Legal', 'HR', 'Product', 'Security')[UNIFORM(0, 4, RANDOM())]::VARCHAR ||
                ' team.'
        ) AS EMBEDDING
    FROM docs d CROSS JOIN chunks c
""").collect()

new_count = session.sql("SELECT COUNT(*) AS CNT FROM DOCUMENT_CHUNKS").collect()[0]["CNT"]
print(f"New row count: {new_count} (added {new_count - baseline} rows)")

In [ ]:
# Trigger a manual refresh to pick up the new rows immediately
session.sql("ALTER CORTEX SEARCH SERVICE DOCUMENT_SEARCH_SVC REFRESH").collect()
print("Manual refresh triggered. The service will process only the new rows.")

In [ ]:
# Verify the new data is searchable
session.sql("""
    SELECT
        value['CHUNK_ID']::TEXT AS CHUNK_ID,
        value['DOC_TYPE']::TEXT AS DOC_TYPE,
        LEFT(value['CHUNK_TEXT']::TEXT, 110) AS CHUNK_TEXT_PREVIEW
    FROM TABLE(FLATTEN(PARSE_JSON(
        SNOWFLAKE.CORTEX.SEARCH_PREVIEW(
            'CORTEX_SEARCH_COST_DEMO.DEMO.DOCUMENT_SEARCH_SVC',
            '{
                "query": "newly added document chunk incrementally",
                "columns": ["CHUNK_ID", "DOC_TYPE", "CHUNK_TEXT"],
                "limit": 3
            }'
        )
    )['results']))
""").show()

### Verify Refresh Was Incremental (Not Full)

`CORTEX_SEARCH_REFRESH_HISTORY` is the definitive way to confirm whether a refresh was INCREMENTAL or FULL. The `REFRESH_ACTION` column tells you exactly what happened, and `INDEX_PREPROCESSING_STATISTICS:numInsertedRows` shows how many rows were processed.

In [ ]:
# Check refresh history: REFRESH_ACTION should be INCREMENTAL (not FULL)
session.sql("""
    SELECT
        NAME,
        DATA_TIMESTAMP,
        REFRESH_ACTION,
        REFRESH_TRIGGER,
        STATE,
        INDEX_PREPROCESSING_STATISTICS:numInsertedRows::INT AS ROWS_INSERTED,
        INDEX_PREPROCESSING_STATISTICS:numDeletedRows::INT AS ROWS_DELETED,
        INDEX_PREPROCESSING_STATISTICS:numCopiedRows::INT AS ROWS_COPIED,
        TIMEDIFF('second', REFRESH_START_TIME, REFRESH_END_TIME) AS DURATION_SEC,
        TARGET_LAG_SEC,
        WAREHOUSE
    FROM TABLE(
        CORTEX_SEARCH_COST_DEMO.INFORMATION_SCHEMA.CORTEX_SEARCH_REFRESH_HISTORY(
            NAME => 'CORTEX_SEARCH_COST_DEMO.DEMO.DOCUMENT_SEARCH_SVC',
            DATA_TIMESTAMP_START => DATEADD('day', -7, CURRENT_TIMESTAMP()),
            RESULT_LIMIT => 20
        )
    )
    ORDER BY DATA_TIMESTAMP DESC
""").show()

In [ ]:
# Summary: count of FULL vs INCREMENTAL vs NO_DATA refreshes
# For a healthy service, scheduled refreshes should be INCREMENTAL or NO_DATA.
# FULL should only appear for CREATION or after an explicit ALTER that forces it.
session.sql("""
    SELECT
        REFRESH_ACTION,
        REFRESH_TRIGGER,
        COUNT(*) AS CNT
    FROM TABLE(
        CORTEX_SEARCH_COST_DEMO.INFORMATION_SCHEMA.CORTEX_SEARCH_REFRESH_HISTORY(
            NAME => 'CORTEX_SEARCH_COST_DEMO.DEMO.DOCUMENT_SEARCH_SVC',
            DATA_TIMESTAMP_START => DATEADD('day', -7, CURRENT_TIMESTAMP()),
            RESULT_LIMIT => 100
        )
    )
    GROUP BY REFRESH_ACTION, REFRESH_TRIGGER
    ORDER BY CNT DESC
""").show()

In [ ]:
# Check cost after incremental refresh
session.sql("""
    SELECT
        USAGE_DATE,
        SERVICE_NAME,
        CONSUMPTION_TYPE,
        CREDITS,
        TOKENS
    FROM SNOWFLAKE.ACCOUNT_USAGE.CORTEX_SEARCH_DAILY_USAGE_HISTORY
    WHERE SERVICE_NAME = 'DOCUMENT_SEARCH_SVC'
    ORDER BY USAGE_DATE DESC
    LIMIT 20
""").show()

---
## Section 5: Optimization Levers

Test the key cost optimization techniques.

In [ ]:
# 1. Enable AUTO_SUSPEND (30 min idle -> suspend serving)
session.sql("""
    ALTER CORTEX SEARCH SERVICE DOCUMENT_SEARCH_SVC
    SET AUTO_SUSPEND = 1800
""").collect()
print("AUTO_SUSPEND set to 1800 seconds (30 minutes).")
print("The service will suspend serving after 30 min of no queries.")

In [ ]:
# 2. Increase TARGET_LAG to reduce refresh frequency
session.sql("""
    ALTER CORTEX SEARCH SERVICE DOCUMENT_SEARCH_SVC
    SET TARGET_LAG = '4 hours'
""").collect()
print("TARGET_LAG increased to 4 hours (was 1 hour).")
print("This reduces warehouse refresh frequency and compute cost.")

In [ ]:
# 3. Increase FULL_INDEX_BUILD_INTERVAL_DAYS for low-churn corpora
session.sql("""
    ALTER CORTEX SEARCH SERVICE DOCUMENT_SEARCH_SVC
    SET FULL_INDEX_BUILD_INTERVAL_DAYS = 7
""").collect()
print("FULL_INDEX_BUILD_INTERVAL_DAYS set to 7 (was default 1).")
print("Index compaction will happen weekly instead of daily.")
print("Note: This does NOT re-embed unchanged data.")

In [ ]:
# 4. Cost projection: what if we archived stale documents?
session.sql("""
    WITH current_stats AS (
        SELECT
            COUNT(*) AS total_rows,
            ROUND(6.3 * COUNT(*) * (768 * 4 + 1000) / 1e9, 2) AS current_monthly_cr
        FROM DOCUMENT_CHUNKS
    ),
    reduced_stats AS (
        SELECT
            COUNT(*) AS active_rows,
            ROUND(6.3 * COUNT(*) * (768 * 4 + 1000) / 1e9, 2) AS reduced_monthly_cr
        FROM DOCUMENT_CHUNKS c
        JOIN DOCUMENTS d ON c.DOC_ID = d.DOC_ID
        WHERE d.STATUS = 'active'
    )
    SELECT
        c.total_rows,
        c.current_monthly_cr,
        r.active_rows,
        r.reduced_monthly_cr,
        c.current_monthly_cr - r.reduced_monthly_cr AS savings_cr,
        ROUND((1 - r.reduced_monthly_cr / NULLIF(c.current_monthly_cr, 0)) * 100, 1) AS pct_savings
    FROM current_stats c, reduced_stats r
""").show()

In [ ]:
# 5. Compare vector dimension impact on serving cost
session.sql("""
    SELECT
        5500 AS row_count,
        768 AS dims_768,
        ROUND(6.3 * 5500 * (768 * 4 + 1000) / 1e9, 4) AS serving_cr_768,
        1536 AS dims_1536,
        ROUND(6.3 * 5500 * (1536 * 4 + 1000) / 1e9, 4) AS serving_cr_1536,
        ROUND(6.3 * 5500 * (1536 * 4 + 1000) / 1e9 / NULLIF(6.3 * 5500 * (768 * 4 + 1000) / 1e9, 0), 2) AS cost_ratio
""").show()

---
## Section 6: Cost Monitoring Dashboard Queries

These queries form a monitoring suite you can schedule or embed in a dashboard.

In [ ]:
# All Cortex Search services in the account with cost breakdown
session.sql("""
    SELECT
        USAGE_DATE,
        DATABASE_NAME || '.' || SCHEMA_NAME || '.' || SERVICE_NAME AS FQN,
        CONSUMPTION_TYPE,
        ROUND(CREDITS, 4) AS CREDITS
    FROM SNOWFLAKE.ACCOUNT_USAGE.CORTEX_SEARCH_DAILY_USAGE_HISTORY
    WHERE USAGE_DATE >= DATEADD('day', -30, CURRENT_DATE())
    ORDER BY USAGE_DATE DESC, FQN, CONSUMPTION_TYPE
    LIMIT 50
""").show()

In [ ]:
# Serving cost trend by service
session.sql("""
    SELECT
        USAGE_DATE,
        SERVICE_NAME,
        ROUND(SUM(CREDITS), 2) AS DAILY_SERVING_CR
    FROM SNOWFLAKE.ACCOUNT_USAGE.CORTEX_SEARCH_DAILY_USAGE_HISTORY
    WHERE CONSUMPTION_TYPE = 'SERVING'
      AND USAGE_DATE >= DATEADD('day', -30, CURRENT_DATE())
    GROUP BY USAGE_DATE, SERVICE_NAME
    ORDER BY USAGE_DATE DESC, DAILY_SERVING_CR DESC
    LIMIT 30
""").show()

In [ ]:
# Inspect service details
session.sql("DESCRIBE CORTEX SEARCH SERVICE DOCUMENT_SEARCH_SVC").show()

In [ ]:
# Production-scale projection: what would 262M rows at 1536 dims cost?
session.sql("""
    SELECT
        262000000 AS row_count,
        1536 AS dims,
        1200 AS payload_bytes,
        ROUND(262000000 * (1536 * 4 + 1200) / 1e9, 1) AS indexed_data_gb,
        ROUND(6.3 * 262000000 * (1536 * 4 + 1200) / 1e9, 0) AS monthly_serving_cr,
        ROUND(6.3 * 262000000 * (1536 * 4 + 1200) / 1e9 / 30, 0) AS daily_serving_cr
""").show()

---
## Summary

In this lab you validated:

1. **BYO embeddings incur zero EMBED_TEXT cost** -- Snowflake indexes user-provided vectors as-is.
2. **Incremental refresh processes only the delta** -- adding 500 rows did not re-process the original 5,000.
3. **CORTEX_SEARCH_REFRESH_HISTORY proves it** -- `REFRESH_ACTION = 'INCREMENTAL'` with `numInsertedRows` matching only the batch size confirms delta-only processing.
4. **Serving cost is proportional to indexed data** -- `6.3 cr/GB/mo` of uncompressed data (vectors + payload).
5. **Optimization levers** -- `AUTO_SUSPEND`, `TARGET_LAG`, corpus pruning, and vector dimension reduction all directly reduce cost.
6. **Cost monitoring** -- `CORTEX_SEARCH_DAILY_USAGE_HISTORY` breaks out SERVING vs. EMBED_TEXT vs. BATCH per service per day.

### Key Takeaways

- Use `PRIMARY KEY` and `REFRESH_MODE = INCREMENTAL` to avoid full rebuilds.
- Never `CREATE OR REPLACE` the source table -- use `INSERT` / `MERGE` to preserve change tracking.
- Use `CORTEX_SEARCH_REFRESH_HISTORY` to audit whether refreshes are truly incremental. Look for `REFRESH_ACTION = 'FULL'` on `REFRESH_TRIGGER = 'SCHEDULED'` -- that means something is forcing full rebuilds.
- `FULL_INDEX_BUILD_INTERVAL_DAYS` compacts the index but does **not** re-embed unchanged data.
- The single biggest cost lever is **indexed data size**: archive stale documents or reduce vector dimensions.
- For teams not ready to change the corpus: `AUTO_SUSPEND`, manual `SUSPEND SERVING`, and matching `TARGET_LAG` to data arrival cadence are the operational levers.

Run `lab/cleanup.sql` to tear down the demo and start fresh.